# Thùng rác phân loại thông minh — Huấn luyện mô hình

**Pipeline:** EDA → làm sạch & tiền xử lý → chia dữ liệu → huấn luyện (MobileNetV3, transfer learning) → đánh giá (10 lớp) → xuất mô hình cho Raspberry Pi 4 (TFLite).

**Dataset:** Garbage Classification V2 (Kaggle, sumn2u) — 10 lớp. Thêm dataset vào notebook qua *Add Input*.

> Bật **GPU (T4/P100)** trong Settings trước khi chạy.

## 0. Cấu hình

In [ ]:
import os, json, random, hashlib, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
import keras
from keras import layers

print("TF", tf.__version__, "| Keras", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

INPUT_ROOT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working/export"); OUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
BATCH = 32
CAP_PER_CLASS = 2500        # giới hạn ảnh mỗi lớp để giảm mất cân bằng (clothes rất nhiều)
SPLIT = (0.70, 0.15, 0.15)  # train / val / test
EPOCHS_HEAD = 10            # giai đoạn 1: đóng băng backbone
EPOCHS_FT = 25              # giai đoạn 2: fine-tune (EarlyStopping sẽ dừng sớm)
LR_HEAD = 1e-3
LR_FT = 1e-4                # 1e-5 quá nhỏ: val gần như không đổi khi fine-tune
UNFREEZE_FRAC = 0.4         # tỉ lệ lớp cuối của backbone được mở khoá
PATIENCE = 5                # EarlyStopping theo val_loss

DATA_ROOT = INPUT_ROOT / "datasets/sumn2u/garbage-classification-v2/original"

CLASSES = ["battery", "biological", "cardboard", "clothes", "glass",
           "metal", "paper", "plastic", "shoes", "trash"]

## 1. Tìm thư mục dataset
Tự dò thư mục chứa đủ các lớp con — không cần sửa đường dẫn tay.

In [ ]:
def find_data_root(root, classes):
    want = set(classes)
    for dirpath, dirnames, _ in os.walk(root):
        if want.issubset({d.lower() for d in dirnames}):
            return Path(dirpath)
    raise FileNotFoundError(f"Không tìm thấy thư mục chứa các lớp {classes} trong {root}")

if not DATA_ROOT.exists():
    DATA_ROOT = find_data_root(INPUT_ROOT, CLASSES)
print("DATA_ROOT =", DATA_ROOT)

# map tên lớp (lowercase) -> thư mục thật
class_dirs = {d.name.lower(): d for d in DATA_ROOT.iterdir() if d.is_dir()}
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

rows = []
for c in CLASSES:
    for f in class_dirs[c].rglob("*"):
        if f.suffix.lower() in IMG_EXT:
            rows.append({"path": str(f), "label": c})
df = pd.DataFrame(rows)
print(len(df), "ảnh")

## 2. EDA

### 2.1 Phân bố lớp

In [ ]:
counts = df["label"].value_counts().reindex(CLASSES)
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x=counts.index, y=counts.values, ax=ax, color="steelblue")
ax.set_title("Số ảnh theo lớp"); ax.tick_params(axis="x", rotation=45)
for i, v in enumerate(counts.values):
    ax.text(i, v, str(v), ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.show()


print("Tỉ lệ mất cân bằng (max/min): %.1f" % (counts.max() / counts.min()))

### 2.2 Kiểm tra ảnh lỗi, kích thước, chế độ màu

In [ ]:
def inspect(path):
    try:
        with Image.open(path) as im:
            w, h = im.size; mode = im.mode
            im.verify()
        return w, h, mode, True
    except Exception:
        return None, None, None, False

info = [inspect(p) for p in df["path"]]
df["w"], df["h"], df["mode"], df["ok"] = zip(*info)
print("Ảnh lỗi:", (~df["ok"]).sum())
print(df.loc[df["ok"], "mode"].value_counts())

ok = df[df["ok"]]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(ok["w"], ok["h"], s=2, alpha=0.3)
axes[0].set_xlabel("width"); axes[0].set_ylabel("height"); axes[0].set_title("Kích thước ảnh")
(ok["w"] / ok["h"]).clip(0, 3).hist(bins=50, ax=axes[1])
axes[1].set_title("Tỉ lệ khung hình w/h")
plt.tight_layout(); plt.show()
print(ok[["w", "h"]].describe().T)

### 2.3 Ảnh trùng lặp (MD5)
Ảnh trùng nằm ở cả train và test sẽ làm điểm test cao ảo → loại bỏ.

In [ ]:
def md5(path, chunk=1 << 16):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

df = df[df["ok"]].copy()
df["md5"] = [md5(p) for p in df["path"]]

dup = df[df.duplicated("md5", keep=False)]
conflict = dup.groupby("md5")["label"].nunique()
print("Ảnh trùng:", len(dup), "| nhóm trùng khác nhãn:", (conflict > 1).sum())

# bỏ ảnh trùng khác nhãn (nhãn mâu thuẫn), còn lại giữ 1 bản
bad = conflict[conflict > 1].index
df = df[~df["md5"].isin(bad)].drop_duplicates("md5").reset_index(drop=True)
print("Còn lại:", len(df))

### 2.4 Ảnh mẫu

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 6, figsize=(12, 2 * len(CLASSES)))
for i, c in enumerate(CLASSES):
    sample = df[df["label"] == c].sample(6, random_state=SEED)["path"]
    for j, p in enumerate(sample):
        axes[i, j].imshow(Image.open(p).convert("RGB")); axes[i, j].axis("off")
    axes[i, 0].set_title(c, loc="left", fontsize=10)
plt.tight_layout(); plt.show()

### 2.5 Độ sáng trung bình theo lớp
Lớp nào có nền/ánh sáng khác biệt rõ thì mô hình dễ “học vẹt” theo nền — cần augmentation độ sáng.

In [ ]:
def brightness(p):
    return np.asarray(Image.open(p).convert("L").resize((64, 64))).mean()

sub = df.groupby("label", group_keys=False).apply(lambda g: g.sample(min(len(g), 200), random_state=SEED))
sub["brightness"] = [brightness(p) for p in sub["path"]]
plt.figure(figsize=(12, 4))
sns.boxplot(data=sub, x="label", y="brightness", order=CLASSES)
plt.xticks(rotation=45); plt.title("Độ sáng trung bình (mẫu 200 ảnh/lớp)"); plt.show()

## 3. Tiền xử lý dữ liệu
Áp dụng cho mọi ảnh, lưu ra `/kaggle/working/clean` (JPEG, chuẩn hoá) để huấn luyện nhanh và nhất quán:
1. **Sửa hướng xoay theo EXIF** (ảnh điện thoại hay bị xoay sai).
2. **Chuyển về RGB**: ảnh xám → RGB; ảnh PNG có kênh trong suốt → ghép lên **nền trắng** (giống nền buồng chụp).
3. **Loại ảnh quá nhỏ** (cạnh ngắn < `MIN_SIDE`) — quá ít thông tin.
4. **Đưa về hình vuông bằng cách đệm nền trắng** (không méo vật thể), resize về `PREP_SIZE`.
5. Lưu JPEG chất lượng 95.

> Trên Pi: cắt ảnh camera thành hình vuông ở giữa rồi resize 224×224, RGB, thang 0..255 (không cần chuẩn hoá thêm — model tự làm).

In [ ]:
from PIL import ImageOps
from concurrent.futures import ThreadPoolExecutor

MIN_SIDE = 64
PREP_SIZE = 256            # lưu lớn hơn IMG_SIZE một chút để augmentation zoom/crop không mất chi tiết
CLEAN_DIR = Path("/kaggle/working/clean")

def preprocess(row):
    src = Path(row["path"])
    dst = CLEAN_DIR / row["label"] / (row["md5"] + ".jpg")
    try:
        with Image.open(src) as im:
            im = ImageOps.exif_transpose(im)
            if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
                im = im.convert("RGBA")
                bg = Image.new("RGB", im.size, (255, 255, 255))
                bg.paste(im, mask=im.split()[-1])
                im = bg
            else:
                im = im.convert("RGB")
            if min(im.size) < MIN_SIDE:
                return None
            im = ImageOps.pad(im, (PREP_SIZE, PREP_SIZE), color=(255, 255, 255),
                              method=Image.Resampling.LANCZOS)
            dst.parent.mkdir(parents=True, exist_ok=True)
            im.save(dst, "JPEG", quality=95)
        return str(dst)
    except Exception:
        return None

t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    df["clean"] = list(ex.map(preprocess, df.to_dict("records")))
print(f"{time.time() - t0:.0f}s")

n_before = len(df)
df = df[df["clean"].notna()].copy()
df["orig_path"], df["path"] = df["path"], df["clean"]
print(f"Giữ {len(df)}/{n_before} ảnh (loại {n_before - len(df)} ảnh quá nhỏ/lỗi)")

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for j, (_, r) in enumerate(df.sample(6, random_state=SEED).iterrows()):
    axes[0, j].imshow(Image.open(r["orig_path"]).convert("RGB")); axes[0, j].set_title("gốc", fontsize=9)
    axes[1, j].imshow(Image.open(r["path"])); axes[1, j].set_title(f"sau xử lý ({r['label']})", fontsize=9)
for ax in axes.flat: ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Chia dữ liệu
Giới hạn `CAP_PER_CLASS` ảnh/lớp, rồi chia **stratified** train/val/test.

In [ ]:
df_cap = (df.groupby("label", group_keys=False)
            .apply(lambda g: g.sample(min(len(g), CAP_PER_CLASS), random_state=SEED))
            .reset_index(drop=True))

tr, va, te = SPLIT
train_df, tmp_df = train_test_split(df_cap, test_size=va + te, stratify=df_cap["label"], random_state=SEED)
val_df, test_df = train_test_split(tmp_df, test_size=te / (va + te), stratify=tmp_df["label"], random_state=SEED)

split_tbl = pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "val": val_df["label"].value_counts(),
    "test": test_df["label"].value_counts(),
}).reindex(CLASSES)
split_tbl.loc["TỔNG"] = split_tbl.sum()
display(split_tbl)

# lưu danh sách chia để tái lập
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    d[["path", "label"]].to_csv(OUT_DIR / f"split_{name}.csv", index=False)  # path = ảnh đã tiền xử lý

## 5. Pipeline dữ liệu (tf.data)

In [ ]:
cls2idx = {c: i for i, c in enumerate(CLASSES)}
AUTOTUNE = tf.data.AUTOTUNE

def load_img(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(img, tf.float32), label   # giữ thang 0..255, model tự tiền xử lý

def make_ds(d, training):
    ds = tf.data.Dataset.from_tensor_slices((d["path"].values, d["label"].map(cls2idx).values))
    if training:
        ds = ds.shuffle(len(d), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_img, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(train_df, True)
val_ds = make_ds(val_df, False)
test_ds = make_ds(test_df, False)

# augmentation vừa phải: mô phỏng ảnh chụp trong thùng (lật ngang, xoay nhẹ, lệch sáng, zoom)
# (bản trước quá mạnh: train acc thấp hơn val acc ~4% -> mô hình bị kìm, học chậm)
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomBrightness(0.15, value_range=(0, 255)),
    layers.RandomContrast(0.15),
], name="augment")

x, _ = next(iter(train_ds))
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(np.clip(augment(x[:1], training=True)[0].numpy(), 0, 255).astype("uint8")); ax.axis("off")
plt.suptitle("Ví dụ augmentation (cùng 1 ảnh)"); plt.show()

## 6. Mô hình
**MobileNetV3-Large** (ImageNet) — nhẹ, chạy tốt trên Pi 4. Tiền xử lý nằm sẵn trong model (`include_preprocessing=True`) nên đầu vào là ảnh RGB 0..255, Pi không cần chuẩn hoá thêm.

In [ ]:
def build_model():
    base = keras.applications.MobileNetV3Large(
        input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False,
        weights="imagenet", include_preprocessing=True)
    base.trainable = False

    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = augment(inp)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(len(CLASSES), activation="softmax", name="probs")(x)
    return keras.Model(inp, out), base

model, base = build_model()
model.summary(show_trainable=True)

cw = compute_class_weight("balanced", classes=np.arange(len(CLASSES)),
                          y=train_df["label"].map(cls2idx).values)
class_weight = dict(enumerate(cw))
print("class_weight:", {CLASSES[k]: round(v, 2) for k, v in class_weight.items()})

## 7. Huấn luyện
### Giai đoạn 1 — chỉ huấn luyện đầu phân loại

In [ ]:
ckpt_path = str(OUT_DIR / "best.keras")

def make_callbacks(best_val_loss=None):
    # tạo MỚI cho mỗi giai đoạn: dùng chung callback giữa 2 lần fit làm giai đoạn 2 mang theo
    # "best" của giai đoạn 1 -> checkpoint không lưu được và EarlyStopping dừng quá sớm
    return [
        keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True,
                                        initial_value_threshold=best_val_loss, verbose=1),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=1),
    ]

model.compile(optimizer=keras.optimizers.Adam(LR_HEAD),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
t0 = time.time()
h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
               class_weight=class_weight, callbacks=make_callbacks(), verbose=1)
print(f"{time.time() - t0:.0f}s")

### Giai đoạn 2 — fine-tune phần cuối backbone
Mở khoá `UNFREEZE_FRAC` lớp cuối, giữ BatchNorm đóng băng, learning rate `LR_FT`. Checkpoint chỉ ghi đè khi val_loss tốt hơn mức tốt nhất của giai đoạn 1.

In [ ]:
base.trainable = True
n_freeze = int(len(base.layers) * (1 - UNFREEZE_FRAC))
for i, l in enumerate(base.layers):
    l.trainable = i >= n_freeze and not isinstance(l, layers.BatchNormalization)
print("Số tham số train được:", sum(int(np.prod(w.shape)) for w in model.trainable_weights))

model.compile(optimizer=keras.optimizers.Adam(LR_FT),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

t0 = time.time()
h2 = model.fit(train_ds, validation_data=val_ds,
               epochs=len(h1.history["loss"]) + EPOCHS_FT, initial_epoch=len(h1.history["loss"]),
               class_weight=class_weight, callbacks=make_callbacks(min(h1.history["val_loss"])), verbose=1)
print(f"{time.time() - t0:.0f}s")

In [ ]:
hist = {k: h1.history[k] + h2.history[k] for k in h1.history}
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, m in zip(axes, ["accuracy", "loss"]):
    ax.plot(hist[m], label="train"); ax.plot(hist["val_" + m], label="val")
    ax.axvline(len(h1.history["loss"]) - 0.5, ls="--", c="gray", label="bắt đầu fine-tune")
    ax.set_title(m); ax.set_xlabel("epoch"); ax.legend()
plt.tight_layout(); plt.show()
hist_df = pd.DataFrame(hist)
hist_df.index = np.arange(1, len(hist_df) + 1); hist_df.index.name = "epoch"
hist_df["phase"] = ["head" if e <= len(h1.history["loss"]) else "fine-tune" for e in hist_df.index]
cols = ["phase", "loss", "accuracy", "val_loss", "val_accuracy"] + (["learning_rate"] if "learning_rate" in hist_df else [])
print("Quá trình huấn luyện:")
show = hist_df[cols].round(4)
if "learning_rate" in show:
    show["learning_rate"] = hist_df["learning_rate"].map("{:.0e}".format)
display(show)
best = hist_df["val_loss"].idxmin()
print(f"Epoch tốt nhất (val_loss nhỏ nhất, model được dùng): {best} ({hist_df.loc[best, 'phase']})"
      f"  |  val_loss = {hist_df.loc[best, 'val_loss']:.4f}  |  val_accuracy = {hist_df.loc[best, 'val_accuracy']:.4f}")
hist_df.to_csv(OUT_DIR / "history.csv")

## 8. Đánh giá trên tập test
### 8.1 Theo 10 lớp

In [ ]:
model = keras.models.load_model(ckpt_path)
probs = model.predict(test_ds, verbose=0)
y_true = test_df["label"].map(cls2idx).values
y_pred = probs.argmax(1)

print("Accuracy 10 lớp: %.4f\n" % accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))

cm = confusion_matrix(y_true, y_pred, normalize="true")
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Dự đoán"); plt.ylabel("Thực tế"); plt.title("Confusion matrix (chuẩn hoá theo hàng)")
plt.show()

### 8.2 Ví dụ dự đoán ngẫu nhiên theo lớp
Mỗi hàng là 1 lớp, 6 ảnh test ngẫu nhiên. Tiêu đề **xanh** = đúng, **đỏ** = sai.

In [ ]:
N_COL = 6
rng = np.random.default_rng()
fig, axes = plt.subplots(len(CLASSES), N_COL, figsize=(2.2 * N_COL, 2.4 * len(CLASSES)))
for r, c in enumerate(CLASSES):
    idx = rng.choice(np.where(y_true == cls2idx[c])[0], N_COL, replace=False)
    for ax, i in zip(axes[r], idx):
        ax.imshow(Image.open(test_df.iloc[i]["path"]))
        ax.set_xticks([]); ax.set_yticks([])
        ok = y_pred[i] == y_true[i]
        ax.set_title(f"T: {CLASSES[y_true[i]]}\nP: {CLASSES[y_pred[i]]} ({probs[i].max():.2f})",
                     fontsize=8, color="green" if ok else "red")
plt.tight_layout(); plt.show()

### 8.3 Các ảnh dự đoán sai tự tin nhất
Ảnh sai nhưng mô hình rất chắc chắn → thường là nhãn sai trong dataset, ảnh mơ hồ, hoặc mô hình học theo nền.

In [ ]:
wrong = np.where(y_pred != y_true)[0]
wrong = wrong[np.argsort(-probs[wrong].max(1))][:12]
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, i in zip(axes.flat, wrong):
    ax.imshow(Image.open(test_df.iloc[i]["path"]).convert("RGB")); ax.axis("off")
    ax.set_title(f"T:{CLASSES[y_true[i]]}\nP:{CLASSES[y_pred[i]]} {probs[i].max():.2f}", fontsize=9)
plt.tight_layout(); plt.show()

## 9. Xuất mô hình cho Raspberry Pi 4
| File | Dùng khi |
|---|---|
| `model.keras` | Huấn luyện tiếp / fine-tune với ảnh tự chụp |
| `model_fp16.tflite` | Chính xác gần như bản gốc, nhẹ ~½ |
| `model_int8.tflite` | **Khuyên dùng trên Pi** nếu là `full-int8` — nhanh nhất. Nếu XNNPACK không chạy được bản full-int8, tự chuyển sang `dynamic-range` (chỉ trọng số int8). Kiểu thực tế ghi trong `labels.json` → `int8_mode` |
| `labels.json` | Thứ tự lớp, kích thước & định dạng đầu vào |

Đầu vào của mọi bản: ảnh **RGB, 224×224, float32, thang 0..255**, shape `(1,224,224,3)`. Đầu ra: softmax 10 lớp.

Bản int8 được xuất với **batch cố định = 1** (Pi chỉ suy luận từng ảnh) — batch động khiến XNNPACK lỗi `failed to prepare` với MobileNetV3 INT8.

In [ ]:
# bản suy luận: bỏ khối augmentation (vốn chỉ chạy khi training) cho gọn graph
inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
x = inp
for l in model.layers[1:]:
    if l.name != "augment":
        x = l(x)
infer_model = keras.Model(inp, x)

infer_model.save(OUT_DIR / "model.keras")
saved_dir = str(OUT_DIR / "saved_model")
infer_model.export(saved_dir)

# FP16
conv = tf.lite.TFLiteConverter.from_saved_model(saved_dir)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.target_spec.supported_types = [tf.float16]
(OUT_DIR / "model_fp16.tflite").write_bytes(conv.convert())

# INT8 — SavedModel riêng với batch cố định = 1 (batch động làm XNNPACK lỗi "failed to prepare").
# Phải đi qua SavedModel (không dùng from_concrete_functions): cách đó để biến ở dạng
# READ_VARIABLE, bước hiệu chỉnh int8 sẽ lỗi "variable != nullptr".
saved_dir_b1 = str(OUT_DIR / "saved_model_b1")
arch = keras.export.ExportArchive()
arch.track(infer_model)
arch.add_endpoint(name="serve", fn=lambda x: infer_model(x, training=False),
                  input_signature=[tf.TensorSpec([1, IMG_SIZE, IMG_SIZE, 3], tf.float32, name="image")])
arch.write_out(saved_dir_b1)

rep_paths = train_df.sample(300, random_state=SEED)["path"].values
def rep_data():
    for p in rep_paths:
        img, _ = load_img(p, 0)
        yield [tf.expand_dims(img, 0)]

def convert_int8(full_int8):
    conv = tf.lite.TFLiteConverter.from_saved_model(saved_dir_b1)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    if full_int8:
        conv.representative_dataset = rep_data   # lượng tử hoá cả activation, I/O vẫn float32
    return conv.convert()

def xnnpack_ok(model_bytes):
    try:
        tf.lite.Interpreter(model_content=model_bytes, num_threads=4).allocate_tensors()
        return True
    except RuntimeError as e:
        print("  XNNPACK lỗi:", str(e).splitlines()[0])
        return False

try:
    int8 = convert_int8(full_int8=True)
    full_ok = xnnpack_ok(int8)
except Exception as e:
    print("  full-int8 chuyển đổi lỗi:", str(e).splitlines()[-1][:200])
    full_ok = False
if full_ok:
    INT8_MODE = "full-int8"
else:
    print("-> chuyển sang dynamic-range (chỉ trọng số int8)")
    int8 = convert_int8(full_int8=False)
    INT8_MODE = "dynamic-range" if xnnpack_ok(int8) else "no-xnnpack"
(OUT_DIR / "model_int8.tflite").write_bytes(int8)
print("model_int8.tflite:", INT8_MODE)

meta = {
    "classes": CLASSES, "img_size": IMG_SIZE, "input": "RGB float32 0..255, shape (1,224,224,3)",
    "int8_mode": INT8_MODE,
}
(OUT_DIR / "labels.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")

for f in sorted(OUT_DIR.glob("*")):
    if f.is_file():
        print(f"{f.name:25s} {f.stat().st_size / 1e6:7.2f} MB")

### 9.1 Kiểm tra lại các file TFLite trên tập test
Đảm bảo lượng tử hoá không làm tụt chính xác; đo thời gian suy luận (CPU Kaggle chỉ để tham khảo — Pi 4 sẽ chậm hơn vài lần).

In [ ]:
def eval_tflite(path, n=None):
    try:
        itp = tf.lite.Interpreter(model_path=str(path), num_threads=4)
        itp.allocate_tensors()
    except RuntimeError:
        print(f"  {path.name}: XNNPACK lỗi -> chạy không có XNNPACK (chậm hơn)")
        itp = tf.lite.Interpreter(
            model_path=str(path), num_threads=4,
            experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)
        itp.allocate_tensors()
    i_det, o_det = itp.get_input_details()[0], itp.get_output_details()[0]
    d = test_df if n is None else test_df.sample(n, random_state=SEED)
    preds, times = [], []
    for p in d["path"]:
        img, _ = load_img(p, 0)
        itp.set_tensor(i_det["index"], img.numpy()[None].astype(i_det["dtype"]))
        t = time.perf_counter(); itp.invoke(); times.append(time.perf_counter() - t)
        preds.append(itp.get_tensor(o_det["index"])[0])
    preds = np.array(preds)
    yt = d["label"].map(cls2idx).values
    return accuracy_score(yt, preds.argmax(1)), np.median(times) * 1000

print(f"{'model':22s} {'acc':>7s} {'ms/img':>7s}")
print(f"{'keras (gốc)':22s} {accuracy_score(y_true, y_pred):7.4f}")
for name in ["model_fp16.tflite", "model_int8.tflite"]:
    acc, ms = eval_tflite(OUT_DIR / name)
    print(f"{name:22s} {acc:7.4f} {ms:7.1f}")
print("int8_mode:", INT8_MODE)

## 10. Dọn dẹp & danh sách file đầu ra
Các file nằm trong `/kaggle/working/export/` — tải ở tab **Output** (hoặc *Save Version* để lưu lâu dài). Ảnh tiền xử lý được xoá vì có thể tạo lại.

In [ ]:
import shutil
shutil.rmtree(CLEAN_DIR, ignore_errors=True)
shutil.rmtree(OUT_DIR / "saved_model_b1", ignore_errors=True)   # chỉ dùng tạm để xuất int8
(OUT_DIR / "best.keras").unlink(missing_ok=True)   # trùng với model.keras

for f in sorted(OUT_DIR.rglob("*")):
    if f.is_file():
        print(f"{str(f.relative_to(OUT_DIR)):45s} {f.stat().st_size / 1e6:7.2f} MB")